# Change in mobility during COVID-19 — paper figures

Reproduces figure_1 .. figure_13 from the flat `data/figure_<i>.csv` files in this folder.

Run this notebook from inside `paper/` (its own directory). If `data/figure_<i>.csv` is missing for a
figure, run `python extract_figure_data.py` first (see `README.md` — that script needs access to the
consolidated result trees on the server, not just this repository checkout).

In [ ]:
import sys
from pathlib import Path
from IPython.display import Image, display

# Make sure the paper/ folder itself (not just its parent) is importable,
# regardless of where Jupyter's working directory ended up.
PAPER_DIR = Path.cwd() if (Path.cwd() / "plot_change_mobility_covid.py").exists() else Path("..")
sys.path.insert(0, str(PAPER_DIR))

import plot_change_mobility_covid as pcmc

pcmc.setup_mpl.setup()
DATA_DIR = PAPER_DIR / "data"
FIG_DIR  = PAPER_DIR / "figures"

def show(name, plot_fn):
    csv_path = DATA_DIR / f"{name}.csv"
    if not csv_path.exists():
        print(f"{csv_path} not found -- run extract_figure_data.py first.")
        return None
    out_path = plot_fn(csv_path, FIG_DIR)
    display(Image(filename=str(out_path)))
    return out_path

## figure_1 — radius of gyration & jump-length distribution (California)

In [ ]:
show("figure_1", pcmc.plot_figure_1)

## figure_2 — k-radius of gyration vs. radius of gyration (California)

In [ ]:
show("figure_2", pcmc.plot_figure_2)

## figure_3 — S(t) exploration curve & location frequency by rank (California)

In [ ]:
show("figure_3", pcmc.plot_figure_3)

## figure_4 — Gonzalez trajectory-shape density (California)

In [ ]:
show("figure_4", pcmc.plot_figure_4)

## figure_5 — sigma_x / sigma_y distributions (California)

In [ ]:
show("figure_5", pcmc.plot_figure_5)

## figure_6 — entropy distributions (California)

In [ ]:
show("figure_6", pcmc.plot_figure_6)

## figure_7 — weekly radius of gyration, by rurality & political party (California)

In [ ]:
show("figure_7", pcmc.plot_figure_7)

## figure_8 — radius of gyration & jump-length distribution (Massachusetts)

In [ ]:
show("figure_8", pcmc.plot_figure_8)

## figure_9 — k-radius of gyration vs. radius of gyration (Massachusetts)

In [ ]:
show("figure_9", pcmc.plot_figure_9)

## figure_10 — S(t) exploration curve & location frequency by rank (Massachusetts)

In [ ]:
show("figure_10", pcmc.plot_figure_10)

## figure_11 — Gonzalez trajectory-shape density (Massachusetts)

In [ ]:
show("figure_11", pcmc.plot_figure_11)

## figure_12 — sigma_x / sigma_y distributions (Massachusetts)

In [ ]:
show("figure_12", pcmc.plot_figure_12)

## figure_13 — entropy distributions (Massachusetts)

In [ ]:
show("figure_13", pcmc.plot_figure_13)

## t_threshold sensitivity tables

The `<feature>_t_threshold_{8,24}_MA-CA.csv` files (also in `data/`) are not figures — they're the
tabular sensitivity-analysis data behind the paper's t_threshold robustness tables. Load them directly
with pandas, e.g.:

In [ ]:
import pandas as pd
pd.read_csv(DATA_DIR / "distance_powerlaw_t_threshold_8_MA-CA.csv")

## Experimental — k-Rg distribution shift vs. the first period

Same grid as figure_2 / figure_9 (periods × k), but rows 2-3 replace the raw per-period
density heatmap with how it changed relative to the pre-restriction baseline
("15 jan - 15 march") on the identical (R_g, k-R_g) bin grid — row 2 is during the
restriction ("15 march - 15 may"), row 3 is after ("15 may - sept"). Row 1 keeps the
baseline heatmap itself for reference. Reads the same `figure_2.csv` / `figure_9.csv` — no
new data file needed.

Two variants of the same idea are shown below, via `pcmc.draw_rg_krg_diff(df, relative=...)`:

1. **Relative (% ) change** (`relative=True`, the default) —
   `(density(period_i) - density(baseline)) / density(baseline)`.
2. **Absolute density diff** (`relative=False`) —
   `density(period_i) - density(baseline)`, in the original PDF units.

Both use a diverging colormap centred at zero: red = more density than baseline, blue =
less. The reason both are worth showing: Rg/k-Rg mass is extremely concentrated in the
single smallest bin (R_g, k-R_g < ~37.5 km) — roughly 70-90% of the *entire* population, in
every period (see the numbers in the Results section below). The absolute diff is
completely dominated by that one cell, so everything else in the grid reads as flat noise.
Dividing by the baseline density in each cell (the relative version) puts every part of the
grid on the same proportional footing, which is what actually reveals the shift — at the
cost of being undefined (left blank) wherever the baseline had zero mass, and of a wide
dynamic range on the positive side (a handful of near-empty baseline cells can swing to
+500-800% off 1-2 samples), which is why its colour scale is capped at the 97th percentile
of observed changes with a ">" arrow for anything clipped above that.

### 1. Relative (%) change vs. baseline

In [ ]:
show("figure_2", pcmc.plot_figure_2_diff)   # California
show("figure_9", pcmc.plot_figure_9_diff)   # Massachusetts

### 2. Absolute density diff vs. baseline (for comparison)

Same rows/columns, `relative=False`: raw `density(period_i) - density(baseline)`, no
normalization by the baseline cell mass.

In [ ]:
show("figure_2", pcmc.plot_figure_2_diff_abs)   # California
show("figure_9", pcmc.plot_figure_9_diff_abs)   # Massachusetts

### Results — what the two views actually show

**Why the absolute-diff panels (§2) look mostly empty/noisy.** In every period, for both
states, the single smallest (R_g, k-R_g) bin (< ~37.5 km on each axis) holds the large
majority of the whole population's probability mass:

| period | California | Massachusetts |
|---|---|---|
| 15 jan – 15 march (baseline) | 73.1% | 73.4% |
| 15 march – 15 may (lockdown) | 88.1% | 92.0% |
| 15 may – sept (reopening) | 65.4% | 71.3% |

That single cell alone accounts for roughly half of the *entire* absolute `|Δ density|`
mass moved between any two periods — so on a linear-vs-density colour scale it saturates
the colorbar and everything else in the 39×39 remaining cells is squeezed into a sliver of
grey. The absolute-diff panels are not wrong, they're just not the right normalization for
a population this concentrated in one bin.

**What the relative-change panels (§1) reveal once that's corrected for:**

- **Lockdown row (15 march – 15 may vs. baseline)** is blue almost everywhere in both
  states: people are more concentrated near-zero R_g than the baseline already was
  (+15-19 points into that one bin), so essentially every other cell in the grid lost
  share. No red band anywhere — a clean, uniform contraction.
- **Reopening row (15 may – sept vs. baseline)** is the more interesting one, and it's
  *not* a simple return to baseline: there's a visible **red band running along/near the
  diagonal at small-to-moderate R_g** (people making short-to-medium trips more often than
  in the pre-COVID baseline), fading into **blue at the largest R_g / k-R_g** (long-distance
  travel stays suppressed relative to baseline even by September). That's consistent with
  a summer mobility rebound concentrated in local/regional trips, with long-distance travel
  lagging behind.
- **California's rebound is stronger than Massachusetts'**: CA's bin-0 share drops from
  73.1% (baseline) to 65.4% (reopening), a 7.7-point fall below its own baseline, while MA
  only drops from 73.4% to 71.3% (2.1 points) — matching the visibly thicker/darker red
  band in the CA reopening panels vs. the thinner one in MA's.
- This holds consistently across `k = 3, 6, 10` in both figures, i.e. the pattern isn't an
  artifact of how many top locations are used to compute k-R_g.

**Caveat that applies to both views:** away from the populated wedge (large R_g with very
small k-R_g, or vice versa), baseline counts are tiny, so both the absolute and — especially
— the relative panels get speckly/noisy there (a cell going from 1 to 2 samples reads as
"+100%"). Read the diagonal band and the near-origin region as signal; treat isolated bright
pixels far from the diagonal as sampling noise, not a real effect.